# VisionDecode — Model Zoo (Models 1–8)

Distorted-text OCR scored by **Character Error Rate (CER)**. This notebook implements and
compares every model in the project plan. 

| # | Model | Backbone | Sequence head | Notes |
|---|-------|----------|---------------|-------|
| 1 | Baseline | EfficientNet-B0 | 6 independent linear heads | reference |
| 2 | CRNN | EfficientNet-B0 | BiGRU ×2 (h=256) | industry standard |
| 3 | Lightweight | MobileNetV3-Small | BiGRU ×2 (h=128) | fast / small |
| 4 | ViT | vit_small_patch16_224 (timm) | 6 heads on pooled token | transformer |
| 5 | Tiny CNN | custom (<500k params) | 6 shared head | real-time CPU |
| 6 | Ensemble | top-3 of the above | per-position vote / avg | hybrid |
| 7 | ResNet | resNet |  gru | reference|
| 8 | LargeMoblieNetV3| MobileNetV3-Large | BiGRU ×2 (h=128) | Fast |


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print('Using device:', device)

Using device: cuda


## Shared infrastructure

Charset, encoding, the CER metric, a dataset, a loader factory, and a reusable
classification-style training / evaluation loop. Every classification model (1, 2, 3, 4, 6)
predicts a fixed `6 × 33` grid and reuses these helpers; TrOCR (5) and the traditional baselines
(8) have their own pipelines.

In [ ]:
CHARS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
NUM_CLASSES = len(CHARS)       
CODE_LEN = 6

char_to_idx = {c: i for i, c in enumerate(CHARS)}
idx_to_char = {i: c for i, c in enumerate(CHARS)}

def encode(text):
    return torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

def decode(indices):
    return ''.join(idx_to_char[int(i)] for i in indices)

print('Classification classes:', NUM_CLASSES, '| code length:', CODE_LEN)

Classification classes: 36 | code length: 6


In [ ]:
def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[m]

def cer(preds, targets):
    edits = sum(levenshtein(p,t) for p,t in zip(preds, targets))
    chars = sum(len(t) for t in targets)
    return edits/max(chars,1)

In [ ]:
DATA_DIR = '../data'

class CaptchaDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df = pd.read_csv(csv_file)
        df = df[df['text'].astype(str).str.len() == CODE_LEN].reset_index(drop=True)
        df = df[df['text'].apply(lambda t: all(c in char_to_idx for c in str(t)))].reset_index(drop=True)
        self.df,self.img_dir,self.transform = df,img_dir, transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir,row['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img,encode(row['text'])

def make_loaders(transform, batch_size=64, val_frac=0.1, seed=42):
    full = CaptchaDataset(os.path.join(DATA_DIR, 'train-labels.csv'),
                          os.path.join(DATA_DIR, 'train_images'), transform)
    val_size = int(val_frac * len(full))
    train, val = random_split(full, [len(full) - val_size, val_size],
                              generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(train,batch_size=batch_size,shuffle=True,num_workers=0)
    vl = DataLoader(val,batch_size=128,shuffle=False, num_workers=0)
    return tl, vl

In [ ]:
def decode_logits(logits):
    idx = logits.argmax(dim=2).cpu()
    return [decode(row) for row in idx]

@torch.no_grad()
def evaluate_cls(model, loader):
    model.eval()
    preds, trues, seq_ok, seq_n = [], [], 0, 0
    for images, labels in loader:
        out = model(images.to(device))           
        p = out.argmax(dim=2).cpu()
        for pi, ti in zip(p, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
        seq_ok += (p == labels).all(dim=1).sum().item()
        seq_n  += labels.size(0)
    return cer(preds, trues), seq_ok / max(seq_n, 1)

def cls_loss(logits, labels):
    return sum(F.cross_entropy(logits[:,p,:], labels[: p]) for p in range(CODE_LEN))

def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, ckpt=None):
    model = model.to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    best = float('inf')
    if ckpt:
        os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = cls_loss(model(images), labels)
            loss.backward(); opt.step()
            running += loss.item() * images.size(0)
        tr = running / len(train_loader.dataset)
        vcer, sacc = evaluate_cls(model, val_loader); sched.step(vcer)
        print(f'Epoch {epoch:2d} | loss {tr:.4f} | val CER {vcer:.4f} | full-code acc {sacc:.4f}')
        if ckpt and vcer < best:
            best = vcer; torch.save(model.state_dict(), ckpt)
            print(f'  ↳ best CER {best:.4f} saved -> {ckpt}')
    return best

imagenet_tf = transforms.Compose([
    transforms.Resize((100, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

## Model 1 — Baseline: EfficientNet-B0 + 6 multi-heads *(reference)*

Carried over from `prototype_model_2.ipynb`. A frozen EfficientNet-B0 trunk (last 3 blocks
fine-tuned) feeds 6 independent classification heads — one per character position. Serves as the
comparison anchor for everything below.

In [ ]:
class CaptchaEfficientNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN):
        super().__init__()
        self.code_len = code_len
        self.model = models.efficientnet_b0(weights='DEFAULT')
        for p in self.model.parameters():
            p.requires_grad = False
        for p in self.model.features[-3:].parameters():
            p.requires_grad = True
        in_features = self.model.classifier[1].in_features
        self.classifiers = nn.ModuleList([
            nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, 256), nn.ReLU(),
                          nn.Dropout(0.3), nn.Linear(256, num_classes))
            for _ in range(code_len)
        ])

    def forward(self, x):
        f = self.model.features(x)
        f = torch.flatten(self.model.avgpool(f), 1)
        return torch.stack([h(f) for h in self.classifiers], dim=1) 

model = CaptchaEfficientNet()
print('Model 1 params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model 1 params: 5178868


In [ ]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model,train_loader,val_loader, epochs=10, lr=1e-3,
                 ckpt='../Artifacts/model1_efficientnet.pth')

Epoch  1 | loss 15.5754 | val CER 0.4991 | full-code acc 0.0045
  ↳ best CER 0.4991 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  2 | loss 8.0265 | val CER 0.3033 | full-code acc 0.1001
  ↳ best CER 0.3033 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  3 | loss 5.2553 | val CER 0.2215 | full-code acc 0.2086
  ↳ best CER 0.2215 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  4 | loss 3.8611 | val CER 0.1793 | full-code acc 0.2866
  ↳ best CER 0.1793 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  5 | loss 3.0374 | val CER 0.1527 | full-code acc 0.3527
  ↳ best CER 0.1527 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  6 | loss 2.4644 | val CER 0.1359 | full-code acc 0.3992
  ↳ best CER 0.1359 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  7 | loss 2.0595 | val CER 0.1208 | full-code acc 0.4487
  ↳ best CER 0.1208 saved -> ../Artifacts/model1_efficientnet.pth
Epoch  8 | loss 1.7282 | val CER 0.1157 | full-code acc 0.4707
  ↳ best CER 0.1157 saved -> ../A

0.10496915124228781

In [9]:
train_loader, val_loader = make_loaders(imagenet_tf)
model = CaptchaEfficientNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model1_efficientnet.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 1 val CER:', vcer, '| full-code acc:', sacc)

Model 1 val CER: 0.10496915124228781 | full-code acc: 0.5027513756878439


## Model 2 — CRNN: EfficientNet-B0 + BiGRU (industry standard)

The EfficientNet feature map is pooled to a width-6 sequence (`AdaptiveAvgPool2d((1, 6))`),
giving 6 timesteps that a 2-layer **bidirectional GRU** (hidden 256) reads left-to-right and
right-to-left before a shared linear layer emits 33 classes per timestep.

*Alternative:* keep the full feature width and train with **CTC loss** for true variable-length
decoding — see `crnn_ctc_model.ipynb` for that variant. Here we use fixed 6-step CE so it stays
directly comparable with the other classification models.

In [ ]:
class CRNN_EfficientNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=256):
        super().__init__()
        self.code_len = code_len
        backbone = models.efficientnet_b0(weights='DEFAULT')
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.features[-3:].parameters():
            p.requires_grad = True
        self.features = backbone.features         
        self.pool = nn.AdaptiveAvgPool2d((1, code_len)) 
        self.gru = nn.GRU(1280, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                       
        f = self.pool(f).squeeze(2)                
        f = f.permute(0, 2, 1)                     
        seq, _ = self.gru(f)                      
        return self.fc(seq)                        

model = CRNN_EfficientNet()
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 6719296


In [ ]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=10, lr=1e-3,
                 ckpt='../Artifacts/model2_crnn_bigru.pth')

Epoch  1 | loss 8.5998 | val CER 0.2567 | full-code acc 0.1601
  ↳ best CER 0.2567 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  2 | loss 3.9353 | val CER 0.1730 | full-code acc 0.3292
  ↳ best CER 0.1730 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  3 | loss 2.6675 | val CER 0.1457 | full-code acc 0.3872
  ↳ best CER 0.1457 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  4 | loss 1.9982 | val CER 0.1306 | full-code acc 0.4327
  ↳ best CER 0.1306 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  5 | loss 1.6009 | val CER 0.1340 | full-code acc 0.4212
Epoch  6 | loss 1.3193 | val CER 0.1157 | full-code acc 0.4662
  ↳ best CER 0.1157 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  7 | loss 1.1561 | val CER 0.1130 | full-code acc 0.4832
  ↳ best CER 0.1130 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  8 | loss 1.0271 | val CER 0.1052 | full-code acc 0.5108
  ↳ best CER 0.1052 saved -> ../Artifacts/model2_crnn_bigru.pth
Epoch  9 | loss 0.9016 | val CER 0.1101 | full-c

0.10521927630481907

In [13]:
model = CRNN_EfficientNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model2_crnn_bigru.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 2 val CER:', vcer, '| full-code acc:', sacc)

Model 2 val CER: 0.10521927630481907 | full-code acc: 0.5107553776888444


## Model 3 — Lightweight: MobileNetV3-Small + BiGRU

Same CRNN recipe as Model 2 but with a **MobileNetV3-Small** trunk (576 feature channels) and a
smaller GRU (hidden 128). Targets fast inference and a small footprint while keeping the sequence
head.

In [ ]:
class CRNN_MobileNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.mobilenet_v3_small(weights='DEFAULT')
        self.features = backbone.features          # output channels: 576
        feat_ch = 576
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                       
        f = self.pool(f).squeeze(2).permute(0, 2, 1)  
        seq, _ = self.gru(f)
        return self.fc(seq)                      

model = CRNN_MobileNet()
print('Model  params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model  params: 2367812


In [ ]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=10, lr=1e-3,
                 ckpt='../Artifacts/model3_mobilenet_gru.pth')

Epoch  1 | loss 10.1624 | val CER 0.3244 | full-code acc 0.0850
  ↳ best CER 0.3244 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  2 | loss 2.6893 | val CER 0.1376 | full-code acc 0.4057
  ↳ best CER 0.1376 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  3 | loss 1.4020 | val CER 0.0939 | full-code acc 0.5523
  ↳ best CER 0.0939 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  4 | loss 0.8893 | val CER 0.0663 | full-code acc 0.6538
  ↳ best CER 0.0663 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  5 | loss 0.6628 | val CER 0.0396 | full-code acc 0.7884
  ↳ best CER 0.0396 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  6 | loss 0.5194 | val CER 0.0395 | full-code acc 0.7834
  ↳ best CER 0.0395 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  7 | loss 0.4135 | val CER 0.0350 | full-code acc 0.8094
  ↳ best CER 0.0350 saved -> ../Artifacts/model3_mobilenet_gru.pth
Epoch  8 | loss 0.3569 | val CER 0.0334 | full-code acc 0.8174
  ↳ best CER 0.0334 saved 

0.028264132066033017

In [16]:
model = CRNN_MobileNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model3_mobilenet_gru.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 3 val CER:', vcer, '| full-code acc:', sacc)

Model 3 val CER: 0.028264132066033017 | full-code acc: 0.8454227113556778


## Model 4 — Vision Transformer (timm `vit_small_patch16_224`)

A pretrained ViT-Small encodes the image (resized to 224×224) into a pooled embedding; 6
classification heads then predict the 6 characters. Uses the `timm` library.

*Notes:* DeiT (`deit_small_patch16_224`) is a drop-in `model_name` swap and is more
data-efficient. To get a true *sequence* output instead of pooled features, set
`num_classes=0, global_pool=''` and attach a head to the patch tokens — kept simple (pooled +
6 heads) here for comparability.

In [ ]:
import timm

class ViTCaptcha(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN,
                 model_name='vit_small_patch16_224'):
        super().__init__()
        self.code_len = code_len
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        feat = self.backbone.num_features         
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(0.2), nn.Linear(feat, num_classes))
            for _ in range(code_len)
        ])

    def forward(self, x):
        f = self.backbone(x)                       
        return torch.stack([h(f) for h in self.heads], dim=1)  

vit_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model = ViTCaptcha()
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

/home/aryansharma/Desktop/Aryan Pendrive Data/ai_projects/.venv/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/home/aryansharma/Desktop/Aryan Pendrive Data/ai_projects/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model params: 21748824


In [ ]:
train_loader, val_loader = make_loaders(vit_tf, batch_size=32)
train_classifier(model, train_loader, val_loader, epochs=10, lr=3e-4,
                 ckpt='../Artifacts/model4_vit.pth')

Epoch  1 | loss 20.7514 | val CER 0.9379 | full-code acc 0.0000
  ↳ best CER 0.9379 saved -> ../Artifacts/model4_vit.pth
Epoch  2 | loss 19.4082 | val CER 0.8037 | full-code acc 0.0000
  ↳ best CER 0.8037 saved -> ../Artifacts/model4_vit.pth


KeyboardInterrupt: 

## Model 4b — Advanced ViTCaptcha (column-strip tokens)

A purpose-built Vision Transformer for this captcha layout, instead of a generic 224×224 ViT.

**Tokenisation idea**
1. Drop **10 px from each side** of the width (200 → **180**) to shed the noisy borders.
2. Slice the 180-wide image into **6 vertical strips of 30 × 100 px** (full height, 30 px wide).
3. Treat **each 30×100 strip as a single token** — a linear patch-embedding flattens it to one
   embedding vector. Six strips → six tokens, one per character column.
4. A small **Transformer encoder** (multi-head self-attention) lets the tokens share context
   (helpful for overlapping / smeared characters), then **each token's output decodes to one
   character** via a shared linear head.

Because the six strips line up with the six characters, the natural mapping is **1 token → 1
character**. The head is easily widened to predict 2 characters per token (set
`chars_per_token=2` and use 3 strips) if a layout ever packs two glyphs per column. Output stays
`(B, 6, 33)`, so it plugs straight into `train_classifier` / `evaluate_cls`.

In [ ]:
class AdvancedViTCaptcha(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, in_ch=3,
                 img_h=100, img_w=200, side_crop=10, strip_w=30,
                 embed_dim=256, depth=4, n_heads=8, mlp_dim=512,
                 dropout=0.1, chars_per_token=1):
        super().__init__()
        self.side_crop = side_crop
        self.strip_w = strip_w
        self.img_h = img_h
        self.chars_per_token = chars_per_token

        cropped_w = img_w - 2 * side_crop                 
        self.num_tokens = cropped_w // strip_w             
        assert cropped_w % strip_w == 0, 'cropped width must divide evenly into strips'
        assert self.num_tokens * chars_per_token == code_len, \
            f'{self.num_tokens} tokens x {chars_per_token} char/token must equal {code_len}'

        token_dim = in_ch * img_h * strip_w               # 3 * 100 * 30 = 9000
        self.patch_embed = nn.Linear(token_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=mlp_dim,
            dropout=dropout, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim,num_classes*chars_per_token)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x[:, :, :, self.side_crop:W - self.side_crop]          
        x = x.unfold(3, self.strip_w, self.strip_w)                
        x = x.permute(0, 3, 1, 2, 4).contiguous()                  
        x = x.flatten(2)                                           

        tok = self.patch_embed(x) + self.pos_embed                 
        tok = self.drop(tok)
        tok = self.encoder(tok)                                  
        tok = self.norm(tok)
        out = self.head(tok)                                       
        return out.view(B, self.num_tokens * self.chars_per_token, -1)  

model = AdvancedViTCaptcha()
_x = torch.randn(2, 3, 100, 200)
print('output shape:', tuple(model(_x).shape))   
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

output shape: (2, 6, 36)
Model params: 4423972


In [ ]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=10, lr=5e-3,
                 ckpt='../Artifacts/model4b_advanced_vit.pth')

Epoch  1 | loss 20.8595 | val CER 0.9695 | full-code acc 0.0000
  ↳ best CER 0.9695 saved -> ../Artifacts/model4b_advanced_vit.pth
Epoch  2 | loss 20.6430 | val CER 0.9676 | full-code acc 0.0000
  ↳ best CER 0.9676 saved -> ../Artifacts/model4b_advanced_vit.pth
Epoch  3 | loss 20.6281 | val CER 0.9673 | full-code acc 0.0000
  ↳ best CER 0.9673 saved -> ../Artifacts/model4b_advanced_vit.pth
Epoch  4 | loss 20.6227 | val CER 0.9695 | full-code acc 0.0000
Epoch  5 | loss 20.6200 | val CER 0.9672 | full-code acc 0.0000
  ↳ best CER 0.9672 saved -> ../Artifacts/model4b_advanced_vit.pth
Epoch  6 | loss 20.6183 | val CER 0.9658 | full-code acc 0.0000
  ↳ best CER 0.9658 saved -> ../Artifacts/model4b_advanced_vit.pth
Epoch  7 | loss 20.6172 | val CER 0.9669 | full-code acc 0.0000
Epoch  8 | loss 20.6158 | val CER 0.9659 | full-code acc 0.0000
Epoch  9 | loss 20.6153 | val CER 0.9673 | full-code acc 0.0000
Epoch 10 | loss 20.6111 | val CER 0.9695 | full-code acc 0.0000


0.9658162414540603

In [21]:
model = AdvancedViTCaptcha().to(device)
model.load_state_dict(torch.load('../Artifacts/model4b_advanced_vit.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 4b val CER:', vcer, '| full-code acc:', sacc)

Model 4b val CER: 0.9658162414540603 | full-code acc: 0.0


## Model 5 — Custom Tiny CNN (ultra lightweight, <500k params)

A from-scratch CNN: 4 conv blocks (Conv→BN→ReLU, with pooling) → `AdaptiveAvgPool2d((1, 6))` →
a single shared linear head applied to each of the 6 timesteps → `6 × 33`. Designed for
real-time CPU inference; parameter count is printed and asserted below 500k.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN):
        super().__init__()
        self.code_len = code_len
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.features = nn.Sequential(
            block(3, 32),  nn.MaxPool2d(2),     
            block(32, 64), nn.MaxPool2d(2),     
            block(64, 128), nn.MaxPool2d(2),    
            block(128, 128),                    
        )
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.head = nn.Linear(128, num_classes)           

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)
        return self.head(f)                         

model = TinyCNN()
n6 = sum(p.numel() for p in model.parameters())
print(f'Model 6 params: {n6:,}')

Model 6 params: 246,180


In [ ]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/model_tinycnn.pth')

Epoch  1 | loss 16.3753 | val CER 0.5086 | full-code acc 0.0000
  ↳ best CER 0.5086 saved -> ../Artifacts/model_tinycnn.pth
Epoch  2 | loss 12.5082 | val CER 0.4991 | full-code acc 0.0005
  ↳ best CER 0.4991 saved -> ../Artifacts/model_tinycnn.pth
Epoch  3 | loss 11.6286 | val CER 0.5392 | full-code acc 0.0010
Epoch  4 | loss 11.1253 | val CER 0.5413 | full-code acc 0.0005
Epoch  5 | loss 10.7231 | val CER 0.5572 | full-code acc 0.0000
Epoch  6 | loss 10.1495 | val CER 0.3696 | full-code acc 0.0090
  ↳ best CER 0.3696 saved -> ../Artifacts/model_tinycnn.pth
Epoch  7 | loss 9.9074 | val CER 0.3830 | full-code acc 0.0075
Epoch  8 | loss 9.6949 | val CER 0.3998 | full-code acc 0.0065
Epoch  9 | loss 9.4960 | val CER 0.4200 | full-code acc 0.0050
Epoch 10 | loss 9.1001 | val CER 0.3776 | full-code acc 0.0135
Epoch 11 | loss 8.9394 | val CER 0.4013 | full-code acc 0.0110
Epoch 12 | loss 8.8254 | val CER 0.3923 | full-code acc 0.0140
Epoch 13 | loss 8.5777 | val CER 0.3696 | full-code acc 0.

KeyboardInterrupt: 

In [24]:
model = TinyCNN().to(device)
model.load_state_dict(torch.load('../Artifacts/model_tinycnn.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 6 val CER:', vcer, '| full-code acc:', sacc)   

Model 6 val CER: 0.36960146740036687 | full-code acc: 0.009004502251125562


## Model 6 — Hybrid / Ensemble (top-3 models)

Combines the per-position class probabilities of the three best classification models by
**soft-voting** (averaging softmax probabilities), then argmax per position. Soft averaging is
more robust than hard majority voting when models disagree with different confidences. A weighted
variant (weights ∝ each model's validation accuracy) is included.

In [ ]:
@torch.no_grad()
def ensemble_predict(models_with_transforms, loader_builder, weights=None):
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    loaders = [loader_builder(tf)[1] for _, tf in models_with_transforms]  # val loaders
    for m, _ in models_with_transforms:
        m.eval().to(device)

    preds, trues = [], []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        labels = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for pi, ti in zip(idx, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
    seq_acc = np.mean([p == t for p, t in zip(preds, trues)])
    return cer(preds, trues), seq_acc, preds

model1 = CaptchaEfficientNet().to(device)
model1.load_state_dict(torch.load('../Artifacts/model1_efficientnet.pth', map_location=device))
model2 = CRNN_EfficientNet().to(device)
model2.load_state_dict(torch.load('../Artifacts/model2_crnn_bigru.pth', map_location=device))
model3 = CRNN_MobileNet().to(device)
model3.load_state_dict(torch.load('../Artifacts/model3_mobilenet_gru.pth', map_location=device))
combo = [(model2, imagenet_tf), (model1, imagenet_tf), (model3, imagenet_tf)]
e_cer, e_acc, _ = ensemble_predict(combo, make_loaders, weights=[0.25, 0.25, 0.5])
print('Ensemble CER:', e_cer, '| full-code acc:', e_acc)

Ensemble CER: 0.014757378689344673 | full-code acc: 0.9169584792396198


## Model 7 CRNN-ResNet Model

This model uses a pretrained ResNet18 backbone (with only layer4 trainable) to extract visual features, followed by adaptive pooling to reshape features into a sequence of length `code_len`. A bidirectional 2-layer GRU processes the sequence, and a linear layer outputs predictions for each time step, making it suitable for sequence recognition tasks like OCR or code generation.

In [ ]:
class CRNN_ResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=256):
        super().__init__()
        self.code_len = code_len
        backbone = models.resnet18(weights='DEFAULT')
        
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.layer4.parameters():
            p.requires_grad = True
        
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(512, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2)
        f = f.permute(0, 2, 1)
        seq, _ = self.gru(f)
        return self.fc(seq)        

model = CRNN_ResNet()
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 10777636


In [ ]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/model7_resnet18.pth')

Epoch  1 | loss 11.9320 | val CER 0.4322 | full-code acc 0.0235
  ↳ best CER 0.4322 saved -> ../Artifacts/model7_resnet18.pth
Epoch  2 | loss 6.0110 | val CER 0.3324 | full-code acc 0.0870
  ↳ best CER 0.3324 saved -> ../Artifacts/model7_resnet18.pth
Epoch  3 | loss 3.8739 | val CER 0.2945 | full-code acc 0.1151
  ↳ best CER 0.2945 saved -> ../Artifacts/model7_resnet18.pth
Epoch  4 | loss 2.6197 | val CER 0.2624 | full-code acc 0.1596
  ↳ best CER 0.2624 saved -> ../Artifacts/model7_resnet18.pth
Epoch  5 | loss 1.8452 | val CER 0.2556 | full-code acc 0.1616
  ↳ best CER 0.2556 saved -> ../Artifacts/model7_resnet18.pth
Epoch  6 | loss 1.3636 | val CER 0.2620 | full-code acc 0.1606
Epoch  7 | loss 1.1084 | val CER 0.2422 | full-code acc 0.1841
  ↳ best CER 0.2422 saved -> ../Artifacts/model7_resnet18.pth
Epoch  8 | loss 0.8884 | val CER 0.2443 | full-code acc 0.1826
Epoch  9 | loss 0.8411 | val CER 0.2480 | full-code acc 0.1776
Epoch 10 | loss 0.6815 | val CER 0.2431 | full-code acc 0.16

0.19476404869101216

In [28]:
model = CRNN_ResNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model7_resnet18.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model ResNet+ BIGRU val CER:', vcer, '| full-code acc:', sacc)

Model ResNet+ BIGRU val CER: 0.19476404869101216 | full-code acc: 0.26263131565782893


## Model 8 CRNN-MobileNet-Advanced Model

This model employs a pretrained MobileNetV3-Large backbone for efficient feature extraction, followed by adaptive pooling to create sequence features of length `code_len`. A deeper 4-layer bidirectional GRU with dropout processes the sequence, making it lightweight yet powerful for sequence recognition tasks.

In [ ]:

class CRNN_MobileNet_Advanced(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.mobilenet_v3_large(weights='DEFAULT')
        self.features = backbone.features
        feat_ch = 960
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                     
        f = self.pool(f).squeeze(2).permute(0, 2, 1)
        seq, _ = self.gru(f)
        return self.fc(seq)


In [30]:
model = CRNN_MobileNet_Advanced().to(device)
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 4707668


In [38]:
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=20, lr=1e-3,
                 ckpt='../Artifacts/model8_mobilenetlarge_gru.pth')

Epoch  1 | loss 0.1392 | val CER 0.0107 | full-code acc 0.9375
  ↳ best CER 0.0107 saved -> ../Artifacts/model8_mobilenetlarge_gru.pth
Epoch  2 | loss 0.1436 | val CER 0.0089 | full-code acc 0.9480
  ↳ best CER 0.0089 saved -> ../Artifacts/model8_mobilenetlarge_gru.pth
Epoch  3 | loss 0.1061 | val CER 0.0086 | full-code acc 0.9500
  ↳ best CER 0.0086 saved -> ../Artifacts/model8_mobilenetlarge_gru.pth
Epoch  4 | loss 0.1224 | val CER 0.0073 | full-code acc 0.9585
  ↳ best CER 0.0073 saved -> ../Artifacts/model8_mobilenetlarge_gru.pth
Epoch  5 | loss 0.1258 | val CER 0.0096 | full-code acc 0.9450
Epoch  6 | loss 0.1320 | val CER 0.0080 | full-code acc 0.9525
Epoch  7 | loss 0.0961 | val CER 0.0097 | full-code acc 0.9435
Epoch  8 | loss 0.0285 | val CER 0.0031 | full-code acc 0.9815
  ↳ best CER 0.0031 saved -> ../Artifacts/model8_mobilenetlarge_gru.pth
Epoch  9 | loss 0.0147 | val CER 0.0030 | full-code acc 0.9820
  ↳ best CER 0.0030 saved -> ../Artifacts/model8_mobilenetlarge_gru.pth
E

0.0017508754377188595

## CRNN-ShuffleNetV2 Model

This model employs a pretrained ShuffleNetV2 backbone for efficient feature extraction with 1024 output channels, followed by adaptive pooling to create sequence features. A 4-layer bidirectional GRU with 0.3 dropout processes the sequence, making it ideal for resource-constrained sequence recognition tasks.

In [ ]:
class CRNN_ShuffleNetV2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.shufflenet_v2_x1_0(weights='DEFAULT')
        
        self.features = nn.Sequential(
            backbone.conv1,
            backbone.maxpool,
            backbone.stage2,
            backbone.stage3,
            backbone.stage4,
            backbone.conv5,
        )
        feat_ch = 1024
        
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=4, batch_first=True,
                          bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                
        f = self.pool(f).squeeze(2)
        f = f.permute(0, 2, 1)
        seq, _ = self.gru(f)
        return self.fc(seq)

In [36]:
model = CRNN_ShuffleNetV2().to(device)
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 3038472


In [37]:
train_classifier(model, train_loader, val_loader, epochs=20, lr=1e-3,
                 ckpt='../Artifacts/model9_shufflenet_gru.pth')

Epoch  1 | loss 10.9551 | val CER 0.1993 | full-code acc 0.2671
  ↳ best CER 0.1993 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  2 | loss 2.2139 | val CER 0.0702 | full-code acc 0.6393
  ↳ best CER 0.0702 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  3 | loss 0.9441 | val CER 0.0428 | full-code acc 0.7694
  ↳ best CER 0.0428 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  4 | loss 0.5660 | val CER 0.0268 | full-code acc 0.8544
  ↳ best CER 0.0268 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  5 | loss 0.4504 | val CER 0.0208 | full-code acc 0.8834
  ↳ best CER 0.0208 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  6 | loss 0.3544 | val CER 0.0183 | full-code acc 0.8939
  ↳ best CER 0.0183 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  7 | loss 0.2787 | val CER 0.0152 | full-code acc 0.9130
  ↳ best CER 0.0152 saved -> ../Artifacts/model9_shufflenet_gru.pth
Epoch  8 | loss 0.2526 | val CER 0.0151 | full-code acc 0.9130
  ↳ best CER 0.0151

0.002751375687843922

## Comparison 

| Model | Val CER ↓   | Full-code acc ↑   | Params | Inference notes |
|-------|-----------|-----------------|--------|-----------------|
| 1 EfficientNet baseline | 0.09763 | 0.5272 | 5,174,242 | reference |
| 2 CRNN + BiGRU | 0.0727 | 0.6373 | 6,717,757 | expected strongest CNN model |
| 3 MobileNet + GRU | 0.01609 | 0.9069 | 1,774,145 | fastest deep model |
| 4 ViT-Small | 0.9648 | 0.0 | 4,423,201 | data-hungry |
| 5 Tiny CNN | 0.3740 | 0.0200 | 245,793 | real-time CPU |
| 6 Ensemble (0.25+0.25+5) | 0.0089 | 0.9480 | — | bester CER |
| 7 ResNet | 0.2001 | 0.2456 | 10,776,097 | untrained baseline |
| 8 MObileNet large + BiGRU | 0.0017 | 0.9895 | 4707668 | Best Acc |
| 9 ShuffleNetV2 + BiGRU | 0.0027 | 0.9835 | 3038472 | Second Best |